# 05 · Sorting & Grouping

This is where pandas starts to feel like "real analysis" — answering questions like
*"what's the total hold amount per region?"* or *"which vendor has the most invoices?"*
This is the pandas equivalent of SQL's `GROUP BY`.


In [1]:
import pandas as pd

df = pd.read_csv("../data/synthetic_ap_invoices.csv", parse_dates=["Invoice Date"])
df['Hold Reason'] = df['Hold Reason'].fillna('Not on Hold')
df.head(2)

,Invoice Number,Vendor Code,Region,Invoice Date,Invoice Amount,Currency,PO Number,Hold Reason,Payment Status
0,US20-INV-200207,V771155,US20,2026-05-09,22435.19,EUR,PO1397710,Not on Hold,On Hold
1,UK30-INV-200191,V431236,UK30,2026-03-03,12379.09,EUR,PO9195810,Not on Hold,On Hold


## Sorting

In [2]:
df.sort_values('Invoice Amount', ascending=False).head()   # largest invoices first

,Invoice Number,Vendor Code,Region,Invoice Date,Invoice Amount,Currency,PO Number,Hold Reason,Payment Status
298,IT25-INV-200144,V818315,IT25,2026-03-04,24992.98,USD,PO1464071,Missing PO,On Hold
172,DE10-INV-200145,V335796,DE10,2026-07-09,24916.43,GBP,PO5381137,Missing GR,On Hold
55,FR15-INV-200088,V832180,FR15,2026-06-09,24825.18,EUR,PO3118324,Not on Hold,On Hold
270,FR15-INV-200003,V449457,FR15,2026-06-01,24764.05,GBP,PO6579079,Missing PO,On Hold
69,UK30-INV-200131,V970910,UK30,2026-05-24,24683.81,USD,PO8599600,Not on Hold,On Hold


In [3]:
df.sort_values(['Region', 'Invoice Amount'], ascending=[True, False]).head(10)
# sort by Region first, then by Invoice Amount descending within each region

,Invoice Number,Vendor Code,Region,Invoice Date,Invoice Amount,Currency,PO Number,Hold Reason,Payment Status
172,DE10-INV-200145,V335796,DE10,2026-07-09,24916.43,GBP,PO5381137,Missing GR,On Hold
195,DE10-INV-200135,V291335,DE10,2026-04-02,24252.37,EUR,PO8954647,Not on Hold,On Hold
262,DE10-INV-200025,V665894,DE10,2026-02-23,23772.59,EUR,PO7569632,Missing GR,On Hold
254,DE10-INV-200125,V249503,DE10,2026-02-26,23328.30,USD,PO8466027,Approval Pending,On Hold
235,DE10-INV-200190,V627035,DE10,2026-04-27,23175.70,USD,PO7733215,Approval Pending,On Hold
144,DE10-INV-200040,V832180,DE10,2026-05-25,22209.45,USD,PO2937878,Missing PO,On Hold
27,DE10-INV-200080,V359178,DE10,2026-04-03,21734.69,EUR,PO9752084,Approval Pending,On Hold
68,DE10-INV-200120,V748143,DE10,2026-06-02,20590.05,GBP,PO8364386,Approval Pending,On Hold
288,DE10-INV-200050,V691723,DE10,2026-04-19,20573.22,USD,PO6499593,Not on Hold,On Hold
183,DE10-INV-200130,V284779,DE10,2026-01-08,20292.25,EUR,PO3977590,Price Difference,On Hold


## `groupby()` — the core of aggregation

Think of it as: **split** the data into groups, **apply** a calculation to each group,
**combine** the results back into a table.


In [4]:
df.groupby('Region')['Invoice Amount'].sum()   # total invoice amount per region

Region
DE10    656401.98
FR15    724513.38
IT25    792532.73
UK30    861521.98
US20    731090.26
Name: Invoice Amount, dtype: float64

In [5]:
df.groupby('Region')['Invoice Amount'].agg(['sum', 'mean', 'count'])
# multiple aggregations at once

,sum,mean,count
Region,,,
DE10,656401.98,11125.457288,59
FR15,724513.38,12491.610000,58
IT25,792532.73,12992.339836,61
UK30,861521.98,13895.515806,62
US20,731090.26,12391.360339,59


In [6]:
df.groupby('Hold Reason').size()   # count of rows per group (like value_counts but more flexible)

Hold Reason
Approval Pending      34
Duplicate Invoice     30
Missing GR            28
Missing PO            37
Not on Hold          109
Price Difference      36
Quantity Mismatch     31
dtype: int64

In [7]:
# Group by MULTIPLE columns
df.groupby(['Region', 'Hold Reason'])['Invoice Amount'].sum()

Region  Hold Reason      
DE10    Approval Pending     121441.29
        Duplicate Invoice     29613.46
        Missing GR            78817.28
        Missing PO            82331.01
        Not on Hold          188065.56
        Price Difference      94768.72
        Quantity Mismatch     61364.66
FR15    Approval Pending      98982.20
        Duplicate Invoice     11226.72
        Missing GR            97447.77
        Missing PO            84798.84
        Not on Hold          302795.69
        Price Difference      26633.54
        Quantity Mismatch    102628.62
IT25    Approval Pending      73489.49
        Duplicate Invoice     84571.38
        Missing GR            69199.17
        Missing PO           146932.74
        Not on Hold          242827.62
        Price Difference     113428.05
        Quantity Mismatch     62084.28
UK30    Approval Pending      86729.36
        Duplicate Invoice     83650.78
        Missing GR            86175.23
        Missing PO            96555.92

## `.agg()` with named aggregations — cleaner, more readable output

This is the pattern you'll actually want in a real report.


In [8]:
summary = df.groupby('Region').agg(
    total_amount=('Invoice Amount', 'sum'),
    avg_amount=('Invoice Amount', 'mean'),
    invoice_count=('Invoice Number', 'count'),
)
summary

,total_amount,avg_amount,invoice_count
Region,,,
DE10,656401.98,11125.457288,60
FR15,724513.38,12491.610000,61
IT25,792532.73,12992.339836,61
UK30,861521.98,13895.515806,62
US20,731090.26,12391.360339,61


## `reset_index()` — turning a grouped result back into a flat DataFrame

`groupby` results often have the group column as the *index*, not a regular column.
`reset_index()` fixes that so you can filter/sort/export normally.


In [9]:
summary = summary.reset_index()
summary.sort_values('total_amount', ascending=False)

,Region,total_amount,avg_amount,invoice_count
3,UK30,861521.98,13895.515806,62
2,IT25,792532.73,12992.339836,61
4,US20,731090.26,12391.360339,61
1,FR15,724513.38,12491.610000,61
0,DE10,656401.98,11125.457288,60


## Exercise

1. Which `Vendor Code` has the highest total `Invoice Amount`? (Hint: `groupby('Vendor Code')['Invoice Amount'].sum().sort_values(ascending=False)`)
2. For each `Hold Reason`, find the average `Invoice Amount` — which hold reason has the highest average?
3. Count invoices per `Region` per `Payment Status` (group by both columns).


In [10]:
# Your code here


---
**Next up:** `02_intermediate/` covers merging datasets, pivot tables, and datetime-based
calculations like invoice aging — the kind of analysis that turns raw data into a report.
